# GenRecon on ULTRRA `va_split` — `genrecon_ultrra.ipynb`

Adapts **[GenRecon](https://github.com/kasothaphie/GenRecon)** to reconstruct a **building** from our Drive **`va_split`** scene (satellite + ground views, ULTRRA-style metadata JSONs), using **Depth Anything 3 (DA3)** for camera poses, intrinsics, and a sparse depth point cloud.

```
va_split/  satellite/images + ground/images  +  jsons/*.json
        |  stage selected views -> run DA3 (poses + depth)
        |  (optional) reorient world so up = +Z
        v
COLMAP text model  rgb/ + colmap/{cameras,images,points3D}.txt
        |  points3D from DA3 depth back-projection (mask-aware if masks exist)
        v
GenRecon  reconstruct_scene.py  --mode Iphone      [conda env `genrecon`]
        v
chunked_to_glb.py  ->  scene.glb   ->   preview + saved to Drive
```

> ### Read this before you run
> 1. **Install is the hard part on Colab.** GenRecon needs Python 3.10 in a separate conda env; DA3 runs in the Colab kernel. First build ~30-60 min; §3 caches the env to Drive.
> 2. **Out-of-domain.** GenRecon checkpoints are indoor-finetuned; aerial/ground building exteriors are a stress test.
> 3. **Merged views.** Satellite and ground pools are combined into one COLMAP model — GenRecon projects all views into a shared grid (no aerial/ground branching at inference).
> 4. **ULTRRA JSONs** in `jsons/` carry geodetic camera locations (`lat/lon/alt`) and sometimes full `omega/phi/kappa` + intrinsics (test-style). Train-style JSONs are often position-only; we therefore default to **DA3** for poses/intrinsics/depth. Set `CAMERA_SOURCE="json"` only if your JSONs include rotation + intrinsics for every selected view.
>
> 5. **Alignment audit.** §5d checks depth reprojection + camera/cloud co-location. Do not run §6 if it FAILs.
>
> **Requires an A100 (80 GB) runtime.**

## 1 — Bootstrap conda (`condacolab`)

GenRecon needs Python 3.10, but the Colab kernel is 3.12. `condacolab.install()` installs Miniconda into `/usr/local` and **restarts the runtime** (this is expected — you'll see a "Your session crashed" style message; that's the restart, not an error).

**Run the cell below once, wait for the restart, then continue from §2.** Do **not** re-run this cell after the restart (it's guarded, but skip it anyway).

In [ ]:
# 1 — Install condacolab (RESTARTS the runtime; run once, then go to §2)
import os, sys
if os.path.exists("/usr/local/conda-meta") or os.path.exists("/opt/conda/conda-meta"):
    print("conda already present -> skip this cell and continue at §2.")
else:
    !pip -q install condacolab
    import condacolab
    condacolab.install()   # <-- triggers a kernel restart; this is normal.

## 2 — Config + mount Drive

Set your paths and knobs here. Everything downstream reads from this cell.

In [ ]:
# 2 — Config + mount Drive
import os, glob, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# Pin numpy<2 BEFORE §3c imports torch (DA3 + pymap3d need this on Colab).
# If you already ran §3c in this session, Runtime → Restart and re-run from here.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4", "scipy==1.13.1"])
print("pinned numpy/scipy for this session")

# ── Input: va_split on Drive (same layout as MV-SAM3D notebooks) ─────────────
DATA_ROOT    = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")  # <-- scene root
SATELLITE_DIR = DATA_ROOT / "satellite"   # expects images/ (+ optional masks/)
GROUND_DIR    = DATA_ROOT / "ground"
JSONS_DIR     = DATA_ROOT / "jsons"         # 00001.json, 00002.json, ...
IMG_SUBDIR    = "images"                    # subfolder under satellite/ and ground/
MASK_SUBDIR   = "masks"

USE_SATELLITE = True                        # include satellite / near-nadir views
USE_GROUND    = True                        # include ground views (merged into one scene)

# ── Cameras / depth source ─────────────────────────────────────────────────
CAMERA_SOURCE = "da3"       # "da3" (default) or "json" (full ULTRRA omega/phi/kappa + intrinsics)
DA3_PROCESS_RES = 1200      # DA3 internal resize (long side); ~800 also worked on 92 views; lower if OOM
DA3_MODEL_ID    = "depth-anything/DA3NESTED-GIANT-LARGE"
FORCE_REDA3     = False

# When True (DA3 path only): Procrustes-align DA3 camera centers to geodetic ENU
# positions from jsons/ (lat/lon/alt). Rotations stay from DA3; depths scaled to match.
ALIGN_DA3_TO_JSON_GEO = True
MIN_GEO_VIEWS_FOR_ALIGN = 3   # need at least this many JSONs with lat/lon/alt

# ── View subset (optional) ─────────────────────────────────────────────────
SELECT_VIEWS  = None          # None = all; or stems/names/indices e.g. ["00001", "00050.jpg", 3]
FILTER_TYPES  = None          # None = all; or e.g. ["airborne", "ground"] from JSON "type"

# ── Up axis ────────────────────────────────────────────────────────────────
UP_AXIS       = "auto"        # "auto", "+x","-x","+y","-y","+z","-z"

# ── Sparse-cloud synthesis (chunking bounds only) ──────────────────────────
PTS_PIX_STRIDE    = 24
PTS_DEPTH_SAMPLES = 3         # range-fallback only (non-DA3 path)
PTS_MAX_TOTAL     = 300_000
USE_MASKS_FOR_CLOUD = True    # back-project only masked building pixels when masks exist
SCENE_XY_CROP = 0.9

# ── GenRecon run knobs ─────────────────────────────────────────────────────
NUM_IMGS_PER_SCENE = 32
CHUNK_SIZE_FACTOR  = 1.08
MIN_POINTS_PER_CHUNK = 30
OCC_THRESHOLD = -3.0
SEED = 42

# ── Working dirs ───────────────────────────────────────────────────────────
WORK       = Path("/content/genrecon_work")
SCENE_DIR  = WORK / "scene"
STAGE_DIR  = WORK / "staged_images"   # flat pool fed to DA3
DA3_DIR    = WORK / "da3"
OUT_DIR    = WORK / "out"
DRIVE_OUT  = Path("/content/drive/MyDrive/genrecon_ultrra_out") / DATA_ROOT.name
ENV_CACHE  = Path("/content/drive/MyDrive/genrecon_env_cache")
for d in (SCENE_DIR, STAGE_DIR, DA3_DIR, OUT_DIR, DRIVE_OUT, ENV_CACHE):
    d.mkdir(parents=True, exist_ok=True)

GENRECON_DIR = Path("/content/GenRecon")
ENV_NAME     = "genrecon"
CKPT_DIR     = ENV_CACHE / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
DA3_ROOT     = Path("/content/Depth-Anything-3")
MVSAM3D_ROOT = Path("/content/MV-SAM3D")   # only for scripts/run_da3.py (same as other notebooks)

TORCH_ARCH = "8.0"
print("Config OK. DATA_ROOT =", DATA_ROOT)
print(f"  satellite={USE_SATELLITE}  ground={USE_GROUND}  camera_source={CAMERA_SOURCE}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3 — Build (or restore) the GenRecon conda env

Creates a `genrecon` env (Python 3.10, torch 2.6 cu124), clones GenRecon `--recursive`, and runs `setup.sh` to compile the CUDA extensions + install flash-attn. **First run: ~30-60 min.** The finished env is tarred to Drive (`ENV_CACHE`) so subsequent sessions restore in a few minutes.

If any single extension fails to compile, the log will show which one — that's the most likely failure point on Colab. Re-running the cell resumes (it skips steps already done).

In [ ]:
# 3a — Clone GenRecon (recursive) — needed both to build and to run
import subprocess, os
from pathlib import Path
if not (GENRECON_DIR / "reconstruct_scene.py").exists():
    !git clone -b main https://github.com/kasothaphie/GenRecon.git --recursive {str(GENRECON_DIR)}
else:
    print("GenRecon already cloned at", GENRECON_DIR)
    # make sure submodules (Trellis.2 etc.) are present
    !cd {str(GENRECON_DIR)} && git submodule update --init --recursive
assert (GENRECON_DIR / "reconstruct_scene.py").exists(), "clone failed"
print("GenRecon files:", sorted(p.name for p in GENRECON_DIR.iterdir())[:20])

# ── Patch a repo bug: IphoneChunker doesn't accept `skip_point_cleaning` even though
#    reconstruct_scene.py passes it (its ScannetIphoneChunker sibling does). Add it. ──
import re
gc = GENRECON_DIR / "inference/get_chunks.py"
src = gc.read_text()
seg = "class IphoneChunker" + src.split("class IphoneChunker", 1)[1]
if "skip_point_cleaning" not in seg[:900]:
    src2 = re.sub(
        r"(min_points_per_chunk: int \| None = None,\n)(\s*\) -> None:)",
        r"\1        skip_point_cleaning: bool = False,\n\2", src, count=1)
    src2 = re.sub(
        r"(\n\s*self\.manual_xy_bounds = manual_xy_bounds\n)",
        r"\1        self.skip_point_cleaning = skip_point_cleaning\n", src2, count=1)
    assert src2 != src and "skip_point_cleaning" in src2, "patch failed — inspect get_chunks.py"
    gc.write_text(src2)
    print("patched IphoneChunker to accept skip_point_cleaning")
else:
    print("IphoneChunker already accepts skip_point_cleaning")

In [ ]:
# 3b — Restore env from Drive cache if available, else build it and cache it.
# Long-running on first build (compiles nvdiffrast/nvdiffrec/CuMesh/FlexGEMM/o-voxel + flash-attn).
import os
ENV_PREFIX = f"/usr/local/envs/{ENV_NAME}"
ENV_TAR    = ENV_CACHE / f"{ENV_NAME}_env.tar.gz"

def env_ok():
    return os.path.exists(f"{ENV_PREFIX}/bin/python")

if env_ok():
    print(f"[env] {ENV_PREFIX} already exists -> reusing.")
elif ENV_TAR.exists():
    print(f"[env] restoring from Drive cache {ENV_TAR} ...")
    os.makedirs(ENV_PREFIX, exist_ok=True)
    !tar -xzf {str(ENV_TAR)} -C {ENV_PREFIX}
    # conda-unpack fixes the hard-coded prefixes if conda-pack was used; harmless otherwise
    !{ENV_PREFIX}/bin/conda-unpack 2>/dev/null || true
    print("[env] restored." if env_ok() else "[env] restore FAILED -> will build fresh below.")

if not env_ok():
    print("[env] building fresh (this is the slow ~30-60 min step) ...")
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
    os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH_ARCH
    build = f'''
set -e
source /usr/local/etc/profile.d/conda.sh
conda create -y -n {ENV_NAME} python=3.10
conda activate {ENV_NAME}
export CUDA_HOME=/usr/local/cuda
export TORCH_CUDA_ARCH_LIST="{TORCH_ARCH}"
pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
cd {str(GENRECON_DIR)}
# basic deps + the 5 CUDA extensions (Trellis.2 build machinery lives in setup.sh)
. ./setup.sh --basic --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm
# prebuilt flash-attn wheel — ABI MUST match torch (pip torch 2.6 is cxx11abi=FALSE). Ampere+; A100=sm80.
ABI=$(python -c "import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')")
pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch2.6cxx11abi${{ABI}}-cp310-cp310-linux_x86_64.whl
# open3d is needed by the chunker's point cleaning path
pip install open3d
'''
    with open("/content/_build_env.sh", "w") as f:
        f.write(build)
    !bash /content/_build_env.sh
    assert env_ok(), "env build failed -> read the log above to see which step broke"
    print("[env] build done -> caching to Drive (slow, one-time) ...")
    !pip -q install conda-pack || true
    !bash -c "source /usr/local/etc/profile.d/conda.sh && conda pack -n {ENV_NAME} -o {str(ENV_TAR)} --ignore-missing-files 2>/dev/null || tar -czf {str(ENV_TAR)} -C {ENV_PREFIX} ."
    print("[env] cached ->", ENV_TAR)

print("env python:", ENV_PREFIX + "/bin/python")

In [ ]:
# 3c — Sanity check: can the env import torch (CUDA) + the compiled extensions?
import subprocess, textwrap
check = textwrap.dedent('''
    import torch, importlib
    print("torch", torch.__version__, "cuda", torch.version.cuda, "avail", torch.cuda.is_available())
    ok = True
    for m in ["flash_attn", "nvdiffrast.torch", "cumesh", "flexgemm", "o_voxel"]:
        try:
            importlib.import_module(m); print("  OK  ", m)
        except Exception as e:
            ok = False; print("  FAIL", m, "->", repr(e)[:160])
    print("ALL EXTENSIONS OK" if ok else "SOME EXTENSIONS MISSING (see FAIL lines)")
''')
r = subprocess.run(["conda", "run", "-n", ENV_NAME, "python", "-c", check],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr[-2000:] if r.stderr else "")

In [ ]:
# 3d — Repair the env: fix flash-attn ABI (+ reinstall FlexGEMM) if §3c showed FAILs.
# Safe to re-run; skips work that's already correct. Run this whenever flash_attn/flexgemm FAIL above.
import subprocess, textwrap

def _run(code):
    r = subprocess.run(["conda", "run", "-n", ENV_NAME, "python", "-c", code],
                       capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip()

# does flash_attn import cleanly?
out, _ = _run("import flash_attn; print('ok')")
if out == "ok":
    print("[flash_attn] already OK.")
else:
    abi, _ = _run("import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')")
    whl = ("https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/"
           f"flash_attn-2.7.3+cu12torch2.6cxx11abi{abi}-cp310-cp310-linux_x86_64.whl")
    print(f"[flash_attn] torch cxx11abi={abi} -> reinstalling matching wheel")
    !conda run -n {ENV_NAME} pip install --no-deps --force-reinstall {whl}

# FlexGEMM: reinstall from a fresh clone if neither import name resolves
out, _ = _run("import importlib;[importlib.import_module(m) for m in ('flexgemm',)];print('ok')")
out2, _ = _run("import importlib;importlib.import_module('flex_gemm');print('ok')")
if out == "ok" or out2 == "ok":
    print("[flexgemm] importable.")
else:
    print("[flexgemm] missing -> reinstalling from source")
    !rm -rf /tmp/extensions/FlexGEMM && git clone https://github.com/JeffreyXiang/FlexGEMM.git /tmp/extensions/FlexGEMM --recursive
    !CUDA_HOME=/usr/local/cuda TORCH_CUDA_ARCH_LIST={TORCH_ARCH} conda run -n {ENV_NAME} pip install /tmp/extensions/FlexGEMM --no-build-isolation

# re-verify + refresh the Drive cache so the fix survives a runtime reset
check = textwrap.dedent('''
    import importlib
    for m in ["flash_attn","nvdiffrast.torch","cumesh","o_voxel","flexgemm","flex_gemm"]:
        try: importlib.import_module(m); print("  OK  ", m)
        except Exception as e: print("  FAIL", m, "->", repr(e)[:110])
''')
o, e = _run(check); print(o); print(e[-400:] if e else "")
print("[cache] refreshing Drive env tarball so the repair persists ...")
!bash -c "source /usr/local/etc/profile.d/conda.sh && conda pack -n {ENV_NAME} -o {str(ENV_TAR)} --force --ignore-missing-files 2>/dev/null || tar -czf {str(ENV_TAR)} -C {ENV_PREFIX} ."
print("[cache] done ->", ENV_TAR)

## 4 — Download checkpoints

Three finetuned generative models: sparse-structure, shape-SLat (512), texture-SLat (512). Cached to Drive so this is a one-time download (~a few GB).

In [ ]:
# 4 — Download GenRecon checkpoints + lay them out with their training config.json
# The loader reads dirname(dirname(ckpt))/config.json, so each .pt must sit at
# <run_dir>/ckpts/<name>.pt next to <run_dir>/config.json (the repo training config).
import os, shutil
from pathlib import Path

# name -> (download url, repo training config that train.py would have written as config.json)
MODELS = {
    "sparse_structure.pt": ("https://kaldir.vc.cit.tum.de/genrecon/sparse_structure.pt",
                            "configs/gen/ss_flow_img/genrecon.json"),
    "shape_slat.pt":       ("https://kaldir.vc.cit.tum.de/genrecon/shape_slat.pt",
                            "configs/gen/slat_flow_img2shape/genrecon_512.json"),
    "texture_slat.pt":     ("https://kaldir.vc.cit.tum.de/genrecon/texture_slat.pt",
                            "configs/gen/slat_flow_imgshape2tex/genrecon_512.json"),
}
MODELS_DIR = Path("/content/genrecon_models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)
ckpt_paths = {}
for name, (url, cfg_rel) in MODELS.items():
    blob = CKPT_DIR / name                                   # cached .pt on Drive
    if blob.exists() and blob.stat().st_size > 1_000_000:
        print(f"  cached: {name} ({blob.stat().st_size/1e6:.0f} MB)")
    else:
        print(f"  downloading {name} ...")
        !wget -q --show-progress -O {str(blob)} {url}
    run = MODELS_DIR / name[:-3]                             # e.g. /content/genrecon_models/sparse_structure
    (run / "ckpts").mkdir(parents=True, exist_ok=True)
    # config.json = the repo training config for this stage
    cfg_src = GENRECON_DIR / cfg_rel
    assert cfg_src.exists(), f"training config missing in repo: {cfg_src}"
    shutil.copy(cfg_src, run / "config.json")
    # link the (Drive-cached) weights into the run layout without re-copying gigabytes
    link = run / "ckpts" / name
    if link.exists() or link.is_symlink(): link.unlink()
    os.symlink(str(blob), str(link))
    ckpt_paths[name] = str(link)

SS_CKPT    = ckpt_paths["sparse_structure.pt"]
SHAPE_CKPT = ckpt_paths["shape_slat.pt"]
TEX_CKPT   = ckpt_paths["texture_slat.pt"]
print("checkpoints laid out with config.json:")
for k in (SS_CKPT, SHAPE_CKPT, TEX_CKPT): print("  ", k)

## 4b — Hugging Face auth (gated DINOv3 encoder) — **required**

GenRecon conditions on **`facebook/dinov3-vitl16-pretrain-lvd1689m`**, which is a **gated** Hugging Face repo. Without access the run dies at "Cannot access gated repo … 401". One-time setup:

1. Open **https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m** while logged in and click **Agree / request access** (Meta grants DINOv3 instantly on accepting the license).
2. Create a **read** token at **https://huggingface.co/settings/tokens**.
3. Run the cell below and paste the token. It writes the token to the shared HF cache + `HF_TOKEN` env var, so the `conda run` subprocess in §6 is authenticated too.

In [ ]:
# 4b — authenticate to Hugging Face + verify access to the gated DINOv3 repo
import os, getpass
try:
    from huggingface_hub import login, hf_hub_download
except ImportError:
    !pip -q install "huggingface_hub>=0.23"
    from huggingface_hub import login, hf_hub_download

# Token resolution order: Colab Secret (key icon) -> env var -> manual prompt.
HF_TOKEN = None
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            HF_TOKEN = userdata.get(k)
            if HF_TOKEN:
                print(f"using Colab Secret '{k}'"); break
        except Exception:
            pass
except Exception:
    pass
HF_TOKEN = (HF_TOKEN or os.environ.get("HF_TOKEN")
            or getpass.getpass("Paste HF token (needs access to facebook/dinov3-...): ")).strip()

os.environ["HF_TOKEN"] = HF_TOKEN                    # inherited by the conda-run subprocess in §6
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("token loaded:", HF_TOKEN[:4] + "…" + HF_TOKEN[-2:] if HF_TOKEN else "NONE")
login(token=HF_TOKEN, add_to_git_credential=False)  # writes /root/.cache/huggingface/token (shared HOME)

DINO_REPO = "facebook/dinov3-vitl16-pretrain-lvd1689m"
try:
    hf_hub_download(DINO_REPO, "config.json", token=HF_TOKEN)
    print(f"OK — access to {DINO_REPO} confirmed. Proceed to §5/§6.")
except Exception as e:
    print("NO ACCESS YET ->", repr(e)[:200])
    print(f"1) Accept the license at https://huggingface.co/{DINO_REPO} (top of page).")
    print("2) If using a Colab Secret, click the key icon (left bar), add HF_TOKEN, and toggle 'Notebook access' ON.")

## 4c — Install Depth Anything 3 (DA3)

Colab bootstrap. Order matters — each dep on its own `pip` line so a single resolver failure can't silently skip the others:
1. clone **Depth-Anything-3** + **MV-SAM3D** (we only use `MV-SAM3D/scripts/run_da3.py`),
2. `pip install addict` **alone**, then hard-verify `import addict` (DA3 imports it but doesn't declare it),
3. `pip install -e Depth-Anything-3` (pulls `pycolmap`, `omegaconf`, `moviepy`, …) + `pymap3d`,
4. `xformers` best-effort (optional — DA3 falls back to pure-torch SwiGLU),
5. pin `numpy==1.26.4`/`scipy==1.13.1` **last** so the §5a `run_da3.py` subprocess loads `numpy<2`,
6. smoke-test `import depth_anything_3`, cache HF weights.

Runs in **this** kernel (not the GenRecon conda env).


In [ ]:
# 4c — Colab DA3 bootstrap
import os, subprocess, sys
from pathlib import Path

def _pip_install(*pkgs, quiet=False):
    """Install into THIS kernel's Python (never bare `pip` / `!pip`)."""
    base = [sys.executable, "-m", "pip", "install"]
    if quiet:
        base.append("-q")
    attempts = [
        base + list(pkgs),
        base + ["--user"] + list(pkgs),
        base + ["--break-system-packages"] + list(pkgs),
    ]
    last_err = None
    for cmd in attempts:
        print("$", " ".join(cmd))
        try:
            subprocess.check_call(cmd)
            return
        except subprocess.CalledProcessError as e:
            last_err = e
    raise RuntimeError(f"pip install failed for {pkgs!r}") from last_err

if CAMERA_SOURCE != "da3":
    print(f"CAMERA_SOURCE={CAMERA_SOURCE!r} -> skip DA3 install (§5 will read JSON metadata).")
    _pip_install("pymap3d", "scipy")
else:
    if not (DA3_ROOT / "src" / "depth_anything_3").exists():
        !git clone --depth 1 https://github.com/ByteDance-Seed/Depth-Anything-3.git {str(DA3_ROOT)}
    else:
        print("DA3 already cloned at", DA3_ROOT)
    if not (MVSAM3D_ROOT / "scripts" / "run_da3.py").exists():
        !git clone --depth 1 https://github.com/devinli123/MV-SAM3D.git {str(MVSAM3D_ROOT)}
    else:
        print("MV-SAM3D already cloned at", MVSAM3D_ROOT)

    print("kernel python:", sys.executable)

    # Re-pin numpy first (DA3 editable install may bump it).
    _pip_install("numpy==1.26.4", "scipy==1.13.1", quiet=True)

    _pip_install("addict")          # DA3 imports this but does not declare it
    import addict
    print("✓ addict", getattr(addict, "__version__", "ok"))

    _pip_install("-e", str(DA3_ROOT), quiet=True)
    _pip_install("pymap3d", quiet=True)

    # xformers optional; re-pin numpy after in case it pulled numpy 2.x
    try:
        import xformers  # noqa: F401
        print("✓ xformers already present")
    except Exception:
        try:
            _pip_install("xformers", quiet=True)
        except Exception as e:
            print("xformers install skipped (optional):", repr(e)[:120])
    _pip_install("numpy==1.26.4", "scipy==1.13.1", quiet=True)

    # Do NOT import DA3 in this kernel if torch was already loaded with numpy 2.x
    # (§3c). Smoke-test in a fresh subprocess instead — same env §5a run_da3.py uses.
    _da3_src = str(DA3_ROOT / "src")
    _smoke = subprocess.run(
        [sys.executable, "-c",
         f"import sys; sys.path.insert(0, {_da3_src!r}); "
         "from depth_anything_3.api import DepthAnything3; print('ok')"],
        capture_output=True, text=True,
    )
    if _smoke.returncode != 0:
        print("------ DA3 import stderr ------")
        print(_smoke.stderr)
        raise RuntimeError(
            "DA3 import failed in subprocess. If you ran §3c before §4c in this session, "
            "Runtime → Restart, then re-run §2 → §4c (numpy must be pinned before torch loads)."
        )
    print("✓ DepthAnything3 import OK (subprocess):", _smoke.stdout.strip())

    try:
        from huggingface_hub import snapshot_download
        da3_cache = snapshot_download(repo_id=DA3_MODEL_ID)
        print("DA3 weights cached at", da3_cache)
    except Exception as e:
        print("snapshot_download failed (will try at inference):", repr(e)[:120])


## 5 — Convert `va_split` → COLMAP text model

Pure numpy / DA3, runs in this kernel. We:
1. collect images from `satellite/` (+ optional `ground/`), match `jsons/<stem>.json`,
2. estimate cameras with **DA3** (w2c, COLMAP convention — **not** inverted) or parse full ULTRRA JSON,
3. optionally **Sim(3)-align** DA3 camera centers to geodetic ENU from JSON (`ALIGN_DA3_TO_JSON_GEO`),
4. reorient world so up = **+Z** (same transform applied to cameras + back-projected points),
5. write COLMAP text + `points3D.txt` using **K_depth** at DA3 resolution for the cloud, **K** at full image resolution for GenRecon.

Run **§5d alignment audit** before §6 — it catches frame/resolution mismatches early.

In [ ]:
# 5a — Collect va_split views, run DA3 (or parse ULTRRA JSON), build per-view records
import os, re, glob, json, shutil, subprocess, sys, numpy as np
from pathlib import Path
from PIL import Image

def img_stem(p):
    return Path(p).stem.replace("_masked", "")

def _natural_sort_key(p):
    stem = Path(p).stem
    try:
        return (0, int(stem), stem)
    except ValueError:
        return (1, 0, stem)

def _resolve_img_dir(mod_root):
    if (mod_root / IMG_SUBDIR).is_dir():
        return mod_root / IMG_SUBDIR
    for cand in ("images", ""):
        d = mod_root / cand if cand else mod_root
        if d.is_dir() and (list(d.glob("*.jpg")) or list(d.glob("*.png"))):
            return d
    raise FileNotFoundError(f"no images under {mod_root}")

def _select(records, select):
    if select is None:
        return records
    keys = set()
    for s in select:
        if isinstance(s, int):
            keys.add(("idx", s))
        else:
            keys.add(("stem", img_stem(str(s))))
    out = []
    for i, r in enumerate(records):
        if ("idx", i) in keys or ("stem", r["stem"]) in keys:
            out.append(r)
    if not out:
        raise ValueError("SELECT_VIEWS matched no images")
    return out

def _json_has_full_pose(meta):
    ext = meta.get("extrinsics", {})
    need = ("lat", "lon", "alt", "omega", "phi", "kappa")
    return all(k in ext for k in need) and "intrinsics" in meta

def _json_has_geo(meta):
    ext = meta.get("extrinsics", {})
    return all(k in ext for k in ("lat", "lon", "alt"))

def _geodetic_origin(metas):
    lats, lons, alts = [], [], []
    for m in metas:
        e = m.get("extrinsics", {})
        if all(k in e for k in ("lat", "lon", "alt")):
            lats.append(e["lat"]); lons.append(e["lon"]); alts.append(e["alt"])
    if not lats:
        return None
    return float(np.mean(lats)), float(np.mean(lons)), float(np.mean(alts))

def _ultrra_json_to_EK(meta, origin):
    """ULTRRA metadata -> (E 4x4 world->cam OpenCV/COLMAP, K 3x3, W, H)."""
    from pymap3d import geodetic2enu
    from scipy.spatial.transform import Rotation as R
    d = meta["extrinsics"]; intr = meta["intrinsics"]
    rot = (
        R.from_matrix([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
        * R.from_euler("zyx", [d["kappa"], d["phi"], d["omega"]], degrees=True).inv()
    )
    tvec = rot.apply(-np.array(geodetic2enu(d["lat"], d["lon"], d["alt"], *origin), dtype=np.float64))
    E = np.eye(4, dtype=np.float64)
    E[:3, :3] = rot.as_matrix(); E[:3, 3] = tvec
    K = np.array([[intr["fx"], 0, intr["cx"]], [0, intr["fy"], intr["cy"]], [0, 0, 1]], np.float64)
    W, H = int(intr["columns"]), int(intr["rows"])
    return E, K, W, H

def _da3_ext_to_E(ext):
    """DA3 prediction.extrinsics are world->cam (w2c), same as COLMAP — do NOT invert."""
    ext = np.asarray(ext, np.float64)
    if ext.shape == (3, 4):
        E = np.eye(4, dtype=np.float64); E[:3, :] = ext; return E
    if ext.shape != (4, 4):
        raise ValueError(f"unexpected DA3 extrinsic shape {ext.shape}")
    return ext.copy()

def _cam_center_from_E(E):
    R, t = E[:3, :3], E[:3, 3]
    return -R.T @ t

def _build_ref_ext_from_geo(v, origin):
    """Synthetic ref w2c: DA3 orientation + JSON geodetic position in ENU."""
    from pymap3d import geodetic2enu
    e = v["meta"]["extrinsics"]
    c2w = np.linalg.inv(v["E"])
    c2w_ref = c2w.copy()
    c2w_ref[:3, 3] = np.array(geodetic2enu(e["lat"], e["lon"], e["alt"], *origin), dtype=np.float64)
    return np.linalg.inv(c2w_ref)

def _ext_to44(ext):
    ext = np.asarray(ext, np.float64)
    if ext.shape[-2:] == (3, 4):
        pad = np.zeros((*ext.shape[:-2], 4, 4), dtype=np.float64)
        pad[..., :3, :4] = ext
        pad[..., 3, 3] = 1.0
        return pad
    return ext

def _umeyama_sim3(src, dst):
    """Sim(3) aligning src -> dst (N,3). Returns R,t,s with dst ≈ s*(R@src.T).T + t."""
    src = np.asarray(src, np.float64)
    dst = np.asarray(dst, np.float64)
    n = src.shape[0]
    mu_s, mu_d = src.mean(0), dst.mean(0)
    src_c, dst_c = src - mu_s, dst - mu_d
    cov = dst_c.T @ src_c / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    var_s = (src_c ** 2).sum() / n
    s = float(np.trace(np.diag(D) @ S) / var_s) if var_s > 1e-12 else 1.0
    t = mu_d - s * (R @ mu_s)
    return R, t, s

def _align_poses_umeyama(ext_ref, ext_est):
    """Umeyama Sim(3) on camera centers (numpy only — avoids evo/scipy in kernel)."""
    pref = np.linalg.inv(_ext_to44(ext_ref))
    pest = np.linalg.inv(_ext_to44(ext_est))
    return _umeyama_sim3(pest[:, :3, 3], pref[:, :3, 3])

def _apply_umeyama_alignment_to_ext(rot, trans, scale, ext_est):
    ext = _ext_to44(ext_est)
    pose = np.linalg.inv(ext)
    R_a = np.einsum("ij,...jk->...ik", rot, pose[..., :3, :3])
    t_a = scale * np.einsum("ij,...j->...i", rot, pose[..., :3, 3]) + trans
    out = np.zeros_like(pose)
    out[..., :3, :3] = R_a
    out[..., :3, 3] = t_a
    out[..., 3, 3] = 1.0
    return np.linalg.inv(out)

# ---- path sanity ----
if USE_SATELLITE and not SATELLITE_DIR.exists():
    raise FileNotFoundError(f"SATELLITE_DIR not found: {SATELLITE_DIR}")
if USE_GROUND and not GROUND_DIR.exists():
    raise FileNotFoundError(f"GROUND_DIR not found: {GROUND_DIR}")
if not JSONS_DIR.is_dir():
    print(f"WARN: JSONS_DIR missing ({JSONS_DIR}) — geo align + type filters unavailable")

# ---- collect image records from satellite / ground ----
raw_records = []
for modality, root, enabled in (
    ("satellite", SATELLITE_DIR, USE_SATELLITE),
    ("ground", GROUND_DIR, USE_GROUND),
):
    if not enabled:
        continue
    idir = _resolve_img_dir(root)
    mask_dir = root / MASK_SUBDIR
    imgs = sorted(glob.glob(str(idir / "*.jpg")) + glob.glob(str(idir / "*.png")))
    print(f"{modality}: {idir} ({len(imgs)} images)")
    for p in imgs:
        st = img_stem(p)
        jp = JSONS_DIR / f"{st}.json"
        meta = json.load(open(jp)) if jp.exists() else {}
        if FILTER_TYPES and meta.get("type") not in FILTER_TYPES:
            continue
        mask_p = None
        if mask_dir.is_dir():
            for ext in (".png", ".jpg", ".jpeg"):
                cand = mask_dir / f"{st}{ext}"
                if cand.exists():
                    mask_p = str(cand); break
        raw_records.append(dict(
            modality=modality, stem=st, name=os.path.basename(p), path=p,
            meta=meta, json_path=str(jp) if jp.exists() else None, mask_path=mask_p,
        ))

assert raw_records, "no images found — check DATA_ROOT / USE_SATELLITE / USE_GROUND"
sel_records = _select(raw_records, SELECT_VIEWS)
n_json = len(list(JSONS_DIR.glob('*.json'))) if JSONS_DIR.is_dir() else 0
print(f"selected {len(sel_records)} views | jsons: {n_json} files in {JSONS_DIR}")

# ---- stage a flat image folder for DA3 (unique names across modalities) ----
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)
_used_names = set()
for r in sel_records:
    staged_name = r["name"]
    if staged_name in _used_names:
        staged_name = f"{r['modality']}_{staged_name}"
    _used_names.add(staged_name)
    staged = STAGE_DIR / staged_name
    shutil.copy(r["path"], staged)
    r["staged_name"] = staged_name
    r["staged_path"] = str(staged)

views = []
geo_align_info = {"applied": False}
use_json_cams = (CAMERA_SOURCE == "json")
if use_json_cams:
    missing = [r["stem"] for r in sel_records if not _json_has_full_pose(r["meta"])]
    if missing:
        raise ValueError(
            f"CAMERA_SOURCE='json' but {len(missing)} views lack omega/phi/kappa+intrinsics, e.g. {missing[:3]}"
        )
    origin = _geodetic_origin([r["meta"] for r in sel_records])
    assert origin is not None, "no geodetic metadata for JSON camera path"
    print(f"[json] ENU origin (lat,lon,alt) = {tuple(round(x, 6) for x in origin)}")
    for r in sel_records:
        E, K, W, H = _ultrra_json_to_EK(r["meta"], origin)
        views.append(dict(**r, E=E, K=K, K_depth=K, W=W, H=H, depth=None, dmin=1.0, dmax=100.0))
else:
    DA3_NPZ = DA3_DIR / "da3_output.npz"
    if FORCE_REDA3 and DA3_NPZ.exists():
        DA3_NPZ.unlink(); print("FORCE_REDA3: deleted cached", DA3_NPZ)
    if not DA3_NPZ.exists():
        if not (MVSAM3D_ROOT / "scripts" / "run_da3.py").exists():
            raise FileNotFoundError(f"run_da3.py missing — re-run §4c ({MVSAM3D_ROOT})")
        _snaps = sorted(
            glob.glob(os.path.expanduser(
                "~/.cache/huggingface/hub/models--depth-anything--DA3NESTED-GIANT-LARGE/snapshots/*/")),
            key=os.path.getmtime, reverse=True,
        )
        da3_model_path = _snaps[0] if _snaps else DA3_MODEL_ID
        print("DA3 model path:", da3_model_path)
        cmd = [
            sys.executable, str(MVSAM3D_ROOT / "scripts" / "run_da3.py"),
            "--image_dir", str(STAGE_DIR),
            "--output_dir", str(DA3_DIR),
            "--model_path", str(da3_model_path),
            "--process_res", str(DA3_PROCESS_RES),
            "--no_vis",
        ]
        print("Running:", " ".join(cmd))
        proc = subprocess.run(cmd, cwd=str(MVSAM3D_ROOT), capture_output=True, text=True)
        if proc.stdout:
            print(proc.stdout)
        if proc.returncode != 0:
            print("------ DA3 STDERR ------")
            print(proc.stderr)
            raise RuntimeError(f"run_da3.py failed (exit {proc.returncode}). See STDERR above.")
        if not DA3_NPZ.exists():
            raise FileNotFoundError(f"run_da3.py finished but {DA3_NPZ} not found")
        print("saved", DA3_NPZ)

    da3 = np.load(DA3_NPZ, allow_pickle=True)
    da3_stems = [Path(str(p)).stem for p in da3["image_files"]]
    da3_by_stem = {s: i for i, s in enumerate(da3_stems)}
    for r in sel_records:
        key = Path(r["staged_name"]).stem
        if key not in da3_by_stem:
            raise KeyError(f"DA3 missing view {key} (have {da3_stems[:5]} ...)")
        j = da3_by_stem[key]
        E = _da3_ext_to_E(da3["extrinsics"][j])
        K_depth = np.asarray(da3["intrinsics"][j], np.float64)
        depth = np.asarray(da3["depth"][j], np.float32)
        W, H = Image.open(r["path"]).size  # PIL (width, height)
        dh, dw = depth.shape  # numpy (height, width)
        K = K_depth.copy()
        if (dw, dh) != (W, H):
            sx, sy = W / dw, H / dh
            K[0, 0] *= sx; K[0, 2] *= sx
            K[1, 1] *= sy; K[1, 2] *= sy
        dvalid = depth[np.isfinite(depth) & (depth > 1e-3)]
        dmin = float(np.percentile(dvalid, 5)) if dvalid.size else 1.0
        dmax = float(np.percentile(dvalid, 95)) if dvalid.size else dmin * 2.0
        views.append(dict(**r, E=E, K=K, K_depth=K_depth, W=W, H=H,
                          depth=depth, dmin=dmin, dmax=dmax))
    print(f"DA3 cameras for {len(views)} views | sample depth range [{views[0]['dmin']:.2f}, {views[0]['dmax']:.2f}]")

    # Optional: Sim(3)-align DA3 poses to geodetic ENU positions from jsons/ (DA3 Umeyama)
    if ALIGN_DA3_TO_JSON_GEO:
        geo_views = [v for v in views if _json_has_geo(v.get("meta", {}))]
        if len(geo_views) >= MIN_GEO_VIEWS_FOR_ALIGN:
            origin = _geodetic_origin([v["meta"] for v in geo_views])
            ext_est = np.stack([v["E"] for v in geo_views], axis=0)
            ext_ref = np.stack([_build_ref_ext_from_geo(v, origin) for v in geo_views], axis=0)
            r, t, s = _align_poses_umeyama(ext_ref, ext_est)
            ext_all = np.stack([v["E"] for v in views], axis=0)
            ext_aligned = _apply_umeyama_alignment_to_ext(r, t, s, ext_all)
            for i, v in enumerate(views):
                v["E"] = ext_aligned[i]
                if v.get("depth") is not None:
                    v["depth"] = v["depth"] * s
                v["dmin"] *= s; v["dmax"] *= s
            ref_centers = np.stack([
                _cam_center_from_E(_build_ref_ext_from_geo(v, origin)) for v in geo_views
            ])
            aln_centers = np.stack([_cam_center_from_E(v["E"]) for v in geo_views])
            err = np.linalg.norm(aln_centers - ref_centers, axis=1)
            geo_align_info = {
                "applied": True, "scale": float(s), "origin": origin,
                "n_views": len(geo_views),
                "center_rmse_m": float(np.sqrt((err ** 2).mean())),
                "center_max_m": float(err.max()),
            }
            print(f"[geo align] DA3 Umeyama -> ENU | scale={s:.4f} | center RMSE={geo_align_info['center_rmse_m']:.2f} m "
                  f"(max {geo_align_info['center_max_m']:.2f} m) over {len(geo_views)} geo JSONs")
        else:
            print(f"[geo align] skipped — only {len(geo_views)} geo JSONs (need {MIN_GEO_VIEWS_FOR_ALIGN})")

types = {}
for v in views:
    t = v.get("meta", {}).get("type", "unknown")
    types[t] = types.get(t, 0) + 1
print("view types:", types)
assert views, "no views built"
print(f"matched {len(views)} views | sample WxH={views[0]['W']}x{views[0]['H']}")


In [ ]:
# 5b — Reorient world so up=+Z, then write cameras.txt + images.txt (+ copy rgb)
import numpy as np, shutil

def cam_center(E):            # camera centre in world = -R^T t
    R, t = E[:3, :3], E[:3, 3]
    return -R.T @ t

def rot_align(a, b):          # 3x3 R with R@a = b  (a,b unit vectors)
    a = a / (np.linalg.norm(a) + 1e-12); b = b / (np.linalg.norm(b) + 1e-12)
    v = np.cross(a, b); c = float(a @ b)
    if np.linalg.norm(v) < 1e-8:
        return np.eye(3) if c > 0 else np.diag([1., -1., -1.])   # parallel / anti-parallel
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * (1.0 / (1.0 + c))

def estimate_up(views):       # plane-normal of camera centres (smallest-variance dir)
    C = np.stack([cam_center(v["E"]) for v in views])
    Cc = C - C.mean(0)
    _, _, Vt = np.linalg.svd(Cc, full_matrices=False)
    n = Vt[-1]
    if n @ np.array([0., 1., 0.]) < 0: n = -n   # deterministic sign; flip via UP_AXIS if inverted
    return n

_AXES = {"+x": [1,0,0], "-x": [-1,0,0], "+y": [0,1,0], "-y": [0,-1,0], "+z": [0,0,1], "-z": [0,0,-1]}
if UP_AXIS == "auto":
    up_vec = estimate_up(views); print(f"[up] auto-estimated up = {np.round(up_vec,3)}")
else:
    up_vec = np.array(_AXES[UP_AXIS], float); print(f"[up] using UP_AXIS={UP_AXIS} -> {up_vec}")
R_align = rot_align(up_vec, np.array([0., 0., 1.]))   # world' = R_align @ world  (up -> +Z)

def quat_from_R(R):           # rotation matrix -> (qw,qx,qy,qz), COLMAP convention
    t = np.trace(R)
    if t > 0:
        s = np.sqrt(t + 1.0) * 2; qw = 0.25 * s
        qx = (R[2,1]-R[1,2])/s; qy = (R[0,2]-R[2,0])/s; qz = (R[1,0]-R[0,1])/s
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s = np.sqrt(1.0+R[0,0]-R[1,1]-R[2,2])*2; qw=(R[2,1]-R[1,2])/s
        qx=0.25*s; qy=(R[0,1]+R[1,0])/s; qz=(R[0,2]+R[2,0])/s
    elif R[1,1] > R[2,2]:
        s = np.sqrt(1.0+R[1,1]-R[0,0]-R[2,2])*2; qw=(R[0,2]-R[2,0])/s
        qx=(R[0,1]+R[1,0])/s; qy=0.25*s; qz=(R[1,2]+R[2,1])/s
    else:
        s = np.sqrt(1.0+R[2,2]-R[0,0]-R[1,1])*2; qw=(R[1,0]-R[0,1])/s
        qx=(R[0,2]+R[2,0])/s; qy=(R[1,2]+R[2,1])/s; qz=0.25*s
    q = np.array([qw,qx,qy,qz]); return q/ (np.linalg.norm(q)+1e-12)

# reoriented extrinsic: E' = [R @ R_align^T | t]
for v in views:
    v["E_o"] = v["E"].copy()
    Rn = v["E"][:3, :3] @ R_align.T
    v["E_r"] = np.eye(4); v["E_r"][:3, :3] = Rn; v["E_r"][:3, 3] = v["E"][:3, 3]

# fresh scene dirs
rgb_dir = SCENE_DIR / "rgb"; colmap_dir = SCENE_DIR / "colmap"
for d in (rgb_dir, colmap_dir):
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True, exist_ok=True)

cam_lines, img_lines = ["# COLMAP cameras (PINHOLE)"], ["# COLMAP images"]
for i, v in enumerate(views, start=1):
    export_name = v.get("staged_name", v["name"])  # unique across satellite+ground
    shutil.copy(v["path"], rgb_dir / export_name)
    K = v["K"]; fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    cam_lines.append(f"{i} PINHOLE {v['W']} {v['H']} {fx:.8f} {fy:.8f} {cx:.8f} {cy:.8f}")
    q = quat_from_R(v["E_r"][:3, :3]); t = v["E_r"][:3, 3]
    img_lines.append(f"{i} {q[0]:.10f} {q[1]:.10f} {q[2]:.10f} {q[3]:.10f} "
                     f"{t[0]:.8f} {t[1]:.8f} {t[2]:.8f} {i} {export_name}")
    img_lines.append("")   # empty POINTS2D line (parser reads 2 lines per image)

(colmap_dir / "cameras.txt").write_text("\n".join(cam_lines) + "\n")
(colmap_dir / "images.txt").write_text("\n".join(img_lines) + "\n")
print(f"wrote cameras.txt + images.txt for {len(views)} views -> {colmap_dir}")
print(f"copied RGB -> {rgb_dir}")

In [ ]:
# 5c — Build a sparse point cloud (points3D.txt) to BOUND + CHUNK the scene.
# DA3 depth + K_depth (same resolution). COLMAP cameras.txt still uses K at full W×H.
import random
import numpy as np, glob, os
from PIL import Image

def _subsample_idx(n, k, seed=0):
    rng = random.Random(seed)
    k = min(k, n)
    return np.fromiter(rng.sample(range(n), k), dtype=np.int64) if k < n else np.arange(n)

def backproject_depthmap(v, dmap, stride, mask=None):
    K = v["K_depth"]
    Einv = np.linalg.inv(v["E_r"])  # c2w
    im = np.asarray(Image.open(v["path"]).convert("RGB"))
    H, W = dmap.shape
    if mask is not None:
        mask = np.asarray(mask)
        if mask.shape[:2] != (H, W):
            mask = np.asarray(Image.fromarray(mask).resize((W, H), Image.NEAREST))
    if im.shape[:2] != (H, W):
        im = np.asarray(Image.fromarray(im).resize((W, H), Image.BILINEAR))
    vs, us = np.mgrid[0:H:stride, 0:W:stride]
    us = us.ravel(); vs = vs.ravel(); d = dmap[vs, us]
    m = np.isfinite(d) & (d > 1e-3)
    if mask is not None:
        m &= mask[vs, us] > 0
    us, vs, d = us[m], vs[m], d[m]
    if us.size == 0:
        return np.empty((0, 3)), np.empty((0, 3))
    pix = np.stack([us, vs, np.ones_like(us)], 0).astype(np.float64)
    cam = (np.linalg.inv(K) @ pix) * d
    world = (Einv[:3, :3] @ cam + Einv[:3, 3:4]).T
    return world, im[vs, us]

def backproject_range(v, stride, depth_fracs, mask=None):
    K = v["K"]
    Einv = np.linalg.inv(v["E_r"])
    im = np.asarray(Image.open(v["path"]).convert("RGB")); H, W = im.shape[:2]
    if mask is not None:
        mask = np.asarray(mask)
        if mask.shape[:2] != (H, W):
            mask = np.asarray(Image.fromarray(mask).resize((W, H), Image.NEAREST))
        vs, us = np.mgrid[0:H:stride, 0:W:stride]
        us = us.ravel(); vs = vs.ravel()
        keep = mask[vs, us] > 0
        us, vs = us[keep], vs[keep]
    else:
        vs, us = np.mgrid[0:H:stride, 0:W:stride]
        us = us.ravel(); vs = vs.ravel()
    rays = np.linalg.inv(K) @ np.stack([us, vs, np.ones_like(us)], 0).astype(np.float64)
    rgb = im[vs, us]; pts, cols = [], []
    for fr in depth_fracs:
        d = v["dmin"] + fr * (v["dmax"] - v["dmin"])
        pts.append((Einv[:3, :3] @ (rays * d) + Einv[:3, 3:4]).T); cols.append(rgb)
    return np.concatenate(pts), np.concatenate(cols)

def _load_mask(v):
    if not USE_MASKS_FOR_CLOUD or not v.get("mask_path"):
        return None
    m = np.asarray(Image.open(v["mask_path"]))
    if m.ndim == 3:
        m = m[..., 0]
    return (m > 0).astype(np.uint8)

depth_fracs = np.linspace(0.2, 0.8, PTS_DEPTH_SAMPLES)
allP, allC = [], []
n_da3, n_range = 0, 0
for v in views:
    mask = _load_mask(v)
    dmap = v.get("depth")
    if dmap is not None:
        P, C = backproject_depthmap(v, dmap, PTS_PIX_STRIDE, mask=mask)
        n_da3 += 1
    else:
        P, C = backproject_range(v, PTS_PIX_STRIDE, depth_fracs, mask=mask)
        n_range += 1
    allP.append(P); allC.append(C)
print(f"point sources: {n_da3} DA3-depth | {n_range} depth-range fallback")
if not allP or all(len(p) == 0 for p in allP):
    raise RuntimeError(
        "no sparse points synthesized — masks may be empty, DA3 depth invalid, or PTS_PIX_STRIDE too large. "
        "Try USE_MASKS_FOR_CLOUD=False, lower PTS_PIX_STRIDE, or check §5d preview."
    )
P = np.concatenate(allP); C = np.concatenate(allC)
if SCENE_XY_CROP and SCENE_XY_CROP < 1.0:
    q = (1.0 - SCENE_XY_CROP) / 2.0
    xlo, xhi = np.percentile(P[:, 0], [100*q, 100*(1-q)])
    ylo, yhi = np.percentile(P[:, 1], [100*q, 100*(1-q)])
    keep = (P[:,0]>=xlo)&(P[:,0]<=xhi)&(P[:,1]>=ylo)&(P[:,1]<=yhi)
    print(f"XY crop (central {SCENE_XY_CROP:.0%}): kept {int(keep.sum())}/{len(P)} points")
    P, C = P[keep], C[keep]
if len(P) > PTS_MAX_TOTAL:
    idx = _subsample_idx(len(P), PTS_MAX_TOTAL, seed=0)
    P, C = P[idx], C[idx]

lines = ["# COLMAP points3D  (ID X Y Z R G B ERROR)"]
for i, (p, c) in enumerate(zip(P, C), start=1):
    lines.append(f"{i} {p[0]:.6f} {p[1]:.6f} {p[2]:.6f} {int(c[0])} {int(c[1])} {int(c[2])} 0.0")
(colmap_dir / "points3D.txt").write_text("\n".join(lines) + "\n")

lo, hi = np.percentile(P, [1, 99], axis=0)
print(f"synthesized {len(P)} points -> points3D.txt")
print(f"bbox (1-99pct)  X[{lo[0]:.1f},{hi[0]:.1f}]  Y[{lo[1]:.1f},{hi[1]:.1f}]  Z[{lo[2]:.1f},{hi[2]:.1f}]")
print(f"Z (vertical) extent = {hi[2]-lo[2]:.1f}  -> chunk_size ~= {(hi[2]-lo[2])*CHUNK_SIZE_FACTOR:.1f}")
_P_preview, _C_preview = P, C


### 5d — Alignment audit (run before GenRecon)

Sanity checks that **cameras, depth point cloud, and COLMAP export share one frame**:
- mean reprojection error of DA3 depth samples (should be small if w2c + K_depth are consistent)
- camera centers vs point-cloud bbox (cameras should surround the building cloud)
- optional geodetic JSON center error after `ALIGN_DA3_TO_JSON_GEO`

In [ ]:
# 5d — Alignment audit (STOP if this looks wrong — do not run §6)
import random
import numpy as np

def _reproj_err_px(v, n_samples=400):
    """Camera-frame roundtrip: depth + K_depth + E_r should reproject to same pixels."""
    dmap = v.get("depth")
    if dmap is None:
        return None
    K = v["K_depth"]
    E = v["E_r"]
    H, W = dmap.shape
    rng = random.Random(0)  # stdlib RNG — np.random breaks if numpy ABI is mixed in kernel
    vs = np.fromiter((rng.randrange(H) for _ in range(n_samples)), dtype=np.int64)
    us = np.fromiter((rng.randrange(W) for _ in range(n_samples)), dtype=np.int64)
    d = dmap[vs, us]
    m = np.isfinite(d) & (d > 1e-3)
    if m.sum() < 10:
        return None
    vs, us, d = vs[m], us[m], d[m]
    pix = np.stack([us, vs, np.ones_like(us)], 0).astype(np.float64)
    cam = (np.linalg.inv(K) @ pix) * d
    c2w = np.linalg.inv(E)
    world = c2w[:3, :3] @ cam + c2w[:3, 3:4]
    proj = E[:3, :3] @ world + E[:3, 3:4]
    uv = (K @ proj)[:2] / np.clip(proj[2], 1e-6, None)
    err = np.linalg.norm(uv - np.stack([us, vs], 0), axis=0)
    return float(np.median(err))

errs = [_reproj_err_px(v) for v in views]
errs_ok = [e for e in errs if e is not None]
_audit_ok = True
if errs_ok:
    mean_e, max_e = float(np.mean(errs_ok)), float(np.max(errs_ok))
    print(f"DA3 depth reprojection (median px / view): mean={mean_e:.3f}  max={max_e:.3f}")
    if mean_e > 2.0:
        print("  FAIL: reprojection too high — poses/intrinsics/depth are inconsistent")
        _audit_ok = False
    else:
        print("  OK: camera-frame depth is consistent with E_r + K_depth")
else:
    print("reprojection check skipped (no DA3 depth)")

C = np.stack([cam_center(v["E_r"]) for v in views])
P = _P_preview
pclo, pchi = np.percentile(P, [5, 95], axis=0)
inside = ((C >= pclo) & (C <= pchi)).mean()
print(f"camera centers inside point-cloud 5-95% bbox (per-axis fraction): {inside:.0%}")
print(f"  cloud span: {np.round(pchi - pclo, 2)}  |  camera spread: {np.round(C.max(0)-C.min(0), 2)}")
if inside < 0.25:
    print("  WARN: cameras mostly outside cloud bbox — try UP_AXIS or USE_MASKS_FOR_CLOUD=False")
    _audit_ok = False

if geo_align_info.get("applied"):
    print(f"geodetic align: RMSE={geo_align_info['center_rmse_m']:.2f} m  scale={geo_align_info['scale']:.4f}")
elif CAMERA_SOURCE == 'da3' and ALIGN_DA3_TO_JSON_GEO:
    print("geodetic align: not applied (insufficient JSON positions)")

diag = np.linalg.norm(pchi - pclo)
dist = np.linalg.norm(C - P.mean(0, keepdims=True), axis=1)
if dist.max() > 3.0 * max(diag, 1e-3):
    print("FAIL: cameras far from point cloud — fix UP_AXIS / masks before §6")
    _audit_ok = False
elif _audit_ok:
    print("PASS: cameras and point cloud are co-located — safe to run §6")

if not _audit_ok:
    print("\n>>> Alignment audit FAILED — fix §5 inputs/flags before running GenRecon.")


In [ ]:
# 5e — PREVIEW: verify the up axis. Camera centres (red) should sit ABOVE the
# building points in the side views (Z up). If it looks tipped over, change UP_AXIS in §2 and re-run §5.
import random
import numpy as np, matplotlib.pyplot as plt

def _subsample_idx(n, k, seed=0):
    rng = random.Random(seed)
    k = min(k, n)
    return np.fromiter(rng.sample(range(n), k), dtype=np.int64) if k < n else np.arange(n)

C_cam = np.stack([cam_center(v["E_r"]) for v in views])
P = _P_preview
sub = P[_subsample_idx(len(P), min(6000, len(P)), seed=1)]
col = _C_preview[_subsample_idx(len(_C_preview), min(6000, len(_C_preview)), seed=1)] / 255.0

fig = plt.figure(figsize=(15, 4.5))
for k, (a, b, ttl) in enumerate([(0, 2, "side X-Z (Z=up)"), (1, 2, "side Y-Z (Z=up)"), (0, 1, "top X-Y")]):
    ax = fig.add_subplot(1, 3, k + 1)
    ax.scatter(sub[:, a], sub[:, b], s=2, c=col, linewidths=0)
    ax.scatter(C_cam[:, a], C_cam[:, b], s=18, c="red", marker="^", label="cameras")
    ax.set_xlabel("xyz"[a]); ax.set_ylabel("xyz"[b]); ax.set_title(ttl); ax.set_aspect("equal")
    if k == 0: ax.legend(fontsize=8)
plt.suptitle(f"Reoriented scene (UP_AXIS={UP_AXIS}) — cameras should be above/around the building in Z")
plt.tight_layout(); plt.show()

## 6 — Run GenRecon (`reconstruct_scene.py --mode Iphone`)

Runs the full generative reconstruction inside the `genrecon` conda env via its CLI. `Iphone` mode reads our `scene/colmap/*.txt` + `scene/rgb/`, chunks the scene, projects the selected views into a shared 3D grid, and generates sparse-structure → shape-SLat → texture-SLat.

Flags tuned for our synthetic cloud: `--skip_point_cleaning` (open3d's metric-tuned filters would delete our arbitrary-scale points) and a low `--min_points_per_chunk`. Outputs `mesh.ply`, `to_glb_inputs.pt`, `chunk_inputs.pt` to `OUT_DIR`.

In [ ]:
# 6 — Run reconstruction (streams live log from the genrecon env)
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH_ARCH
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set -> run §4b first (gated DINOv3 access is required)."

# pick a pipeline config that exists (texture = PBR output; falls back to original)
pcfg = GENRECON_DIR / "configs/pipelines/texture.json"
if not pcfg.exists(): pcfg = GENRECON_DIR / "configs/pipelines/original.json"
pcfg_arg = f"--pipeline_config {pcfg}" if pcfg.exists() else ""

cmd = (
    f"cd {GENRECON_DIR} && conda run -n {ENV_NAME} --no-capture-output python reconstruct_scene.py "
    f"--mode Iphone "
    f"--path {SCENE_DIR} "
    f"--output_path {OUT_DIR} "
    f"--ss_ckpt {SS_CKPT} --shape_ckpt {SHAPE_CKPT} --tex_ckpt {TEX_CKPT} "
    f"--colmap_subdir colmap "
    f"--num_imgs_per_scene {NUM_IMGS_PER_SCENE} "
    f"--chunk_size_factor {CHUNK_SIZE_FACTOR} "
    f"--min_points_per_chunk {MIN_POINTS_PER_CHUNK} "
    f"--occ_threshold {OCC_THRESHOLD} "
    f"--skip_point_cleaning "
    f"--proj_batch_voxels 2048 "
    f"--seed {SEED} "
    f"--save_imgs "
    f"{pcfg_arg}"
)
print(cmd, "\n" + "=" * 80)
!{cmd}
print("=" * 80)
print("outputs:", sorted(p.name for p in OUT_DIR.iterdir()) if OUT_DIR.exists() else "NONE")

In [ ]:
# 6b — Diagnostics: what did GenRecon actually condition on? (works even if §6 crashed
# in generation — these are written during image selection). Confirms the building is in-frame
# and the chunks sit on it. If the cond views are sky/ground or the layout misses the building,
# the up axis / chunking is off (fix UP_AXIS in §2) rather than OCC_THRESHOLD.
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

def _grid(paths, title, titles=None, ncol=4):
    if not paths: return
    n = len(paths); ncol = min(n, ncol); nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4 * nrow), squeeze=False)
    ax = axes.ravel()
    for i, p in enumerate(paths):
        ax[i].imshow(Image.open(p)); ax[i].axis("off")
        ax[i].set_title(titles[i] if titles else "", fontsize=9)
    for j in range(n, len(ax)): ax[j].axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

lay = OUT_DIR / "chunk_layout.png"
if lay.exists():
    plt.figure(figsize=(6, 6)); plt.imshow(Image.open(lay)); plt.axis("off")
    plt.title("chunk layout (top-down over synth cloud)"); plt.show()

cond = sorted(OUT_DIR.glob("chunk_*/cond2d.png"))
_grid(cond, "per-chunk 2D conditioning view (should show the building)",
      titles=[p.parent.name for p in cond])
scene_imgs = sorted((OUT_DIR / "scene").glob("view_*.png"))[:8]
_grid(scene_imgs, "sample of the scene views fed to the generator")
print("chunks:", len(cond), "| scene views saved:", len(scene_imgs))

## 7 — Bake `scene.glb` + preview + save to Drive

`chunked_to_glb.py` fuses the per-chunk outputs into a single textured `scene.glb`. We then preview it and copy the results to `DRIVE_OUT`.

In [ ]:
# 7a — Bake scene.glb
cmd = (
    f"cd {GENRECON_DIR} && conda run -n {ENV_NAME} --no-capture-output python chunked_to_glb.py "
    f"--inputs {OUT_DIR}/to_glb_inputs.pt "
    f"--chunk_inputs {OUT_DIR}/chunk_inputs.pt "
    f"--output_dir {OUT_DIR}"
)
print(cmd, "\n" + "=" * 80)
!{cmd}
glb = OUT_DIR / "scene.glb"
print("scene.glb exists:", glb.exists(), "| size:", (glb.stat().st_size/1e6 if glb.exists() else 0), "MB")

In [ ]:
# 7b — Preview the mesh + chunk layout, then copy results to Drive
import shutil, sys, subprocess, importlib, numpy as np, matplotlib.pyplot as plt
try:
    import trimesh
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh"])
    importlib.invalidate_caches()
    import trimesh

# chunk layout (top-down) that GenRecon saved
layout = OUT_DIR / "chunk_layout.png"
if layout.exists():
    from PIL import Image as _I
    plt.figure(figsize=(6, 6)); plt.imshow(_I.open(layout)); plt.axis("off")
    plt.title("chunk layout"); plt.show()

glb = OUT_DIR / "scene.glb"
if glb.exists():
    scene = trimesh.load(str(glb), force="scene")
    mesh = trimesh.util.concatenate([g for g in scene.geometry.values()])
    print(f"scene.glb: {len(mesh.vertices)} verts / {len(mesh.faces)} faces")
    V, F = np.asarray(mesh.vertices), np.asarray(mesh.faces)
    if len(F) > 120_000:                      # decimate for a fast matplotlib preview
        import random
        keep = np.fromiter(random.Random(0).sample(range(len(F)), 120_000), dtype=np.int64)
        F = F[keep]
    fig = plt.figure(figsize=(16, 5))
    for i, (az, el) in enumerate([(-60, 20), (30, 20), (120, 20)]):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        ax.plot_trisurf(V[:, 0], V[:, 1], V[:, 2], triangles=F, cmap="viridis", linewidth=0, antialiased=False)
        ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1)); ax.set_title(f"scene.glb ({az},{el})")
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    plt.suptitle("GenRecon reconstruction"); plt.tight_layout(); plt.show()
else:
    print("no scene.glb — check the §6/§7a logs.")

# copy everything to Drive
for p in OUT_DIR.glob("*"):
    dst = DRIVE_OUT / p.name
    if p.is_dir():
        shutil.copytree(p, dst, dirs_exist_ok=True)
    else:
        shutil.copy(p, dst)
print("copied results ->", DRIVE_OUT)
print("Download scene.glb and open in any 3D viewer for the textured result.")

## Notes & troubleshooting

- **`max(): ... input.numel() == 0` in shape-SLat.** Lower `OCC_THRESHOLD` in §2 (e.g. -4 / -5). Check §5d preview first.
- **"Cannot access gated repo … 401" for `facebook/dinov3-...`.** Do §4b: accept the license, paste a read token.
- **Install failed on one extension.** Check §3b log; re-run — it resumes from Drive cache.
- **Building looks sideways / over-chunked.** Fix `UP_AXIS` in §2 and re-run §5.
- **Empty chunks.** Lower `MIN_POINTS_PER_CHUNK`, densify cloud (`PTS_PIX_STRIDE`), ensure masks exist under `satellite/masks` and `ground/masks`.
- **DA3 vs JSON cameras.** Default `CAMERA_SOURCE="da3"` because train-style ULTRRA JSONs are often position-only. Use `CAMERA_SOURCE="json"` only when every selected `jsons/*.json` has `omega/phi/kappa` and `intrinsics` (test-style metadata).
- **OOM.** Lower `NUM_IMGS_PER_SCENE` (e.g. 16); `--proj_batch_voxels 2048` already set in §6.
- **Local dry-run.** Point `DATA_ROOT` at a local copy of `va_split` (or symlink) with the same `satellite/`, `ground/`, `jsons/` layout.- **Alignment / sideways building.** §5d prints reprojection error + camera/cloud co-location. If reprojection > ~2 px or cameras sit outside the cloud, re-run §5 (don't proceed to §6). Toggle `ALIGN_DA3_TO_JSON_GEO` if JSON geodetic positions should anchor the DA3 frame.
